# 01 — LLM local (comptes-rendus)

**Emplacement** : `03_llm/01_llm_local_cr.ipynb` · modèle **distinct** du score tabulaire.

| Modèle | Rôle |
|--------|------|
| Qwen2.5 (GGUF) | Brief CR masqué, local (llama.cpp) |
| TF-IDF + logistique | Fallback / entraînement texte |

Garde-fous : masquage identité, prompt anti-hallucination, bascule extractive si > 15 % d'invention.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
import json
from src.data.paths import ProjectPaths, get_project_root
import pandas as pd
paths = ProjectPaths(root=get_project_root())
print(json.dumps(json.loads((paths.models/'cr_llm_rapport.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
pd.read_csv(paths.models/'cr_justifications_echantillon.csv').head(3)

{
  "tfidf": {
    "modele": "tfidf_logistique",
    "justification": "Modèle de langue statistique entraîné localement sur CR masqués. Adapté à n faible et textes courts (~180 caractères). Split patient identique au modèle tabulaire.",
    "val": {
      "roc_auc": 0.6730462519936204,
      "pr_auc": 0.33671494512357625,
      "precision": 0.20833333333333334,
      "recall": 0.9090909090909091,
      "f2": 0.5434782608695652,
      "threshold": 0.37499999999999994,
      "n": 68,
      "prevalence": 0.16176470588235295
    },
    "test": {
      "roc_auc": 0.4453125,
      "pr_auc": 0.31076731332710694,
      "precision": 0.21428571428571427,
      "recall": 0.75,
      "f2": 0.5,
      "threshold": 0.37499999999999994,
      "n": 72,
      "prevalence": 0.2222222222222222
    }
  },
  "llm_backend": "llama.cpp",
  "n_echantillon": 20,
  "fuite_identite_n": 0,
  "hallucination_moyenne": 0.007670454545454545
}


,SejourID,backend,justification,fuite_identite,taux_hallucination
0,SEJ-000001,llama.cpp,Patient admis pour hémorragie digestive. Antéc...,False,0.0
1,SEJ-000002,llama.cpp,Séjour de 3j. Motif : chute du sujet âgé. Surv...,False,0.0
2,SEJ-000004,llama.cpp,Séjour de 21j. Motif : décompensation diabétiq...,False,0.0


## Lecture des résultats
- Le score TF-IDF sur texte seul est **faible** (AUC test ~0.45) : les CR synthétiques sont stéréotypés et peu discriminants. On le garde comme **complément**, pas comme score principal.
- `fuite_identite_n = 0` : le coffre n'apparaît pas dans les justifications.
- Hallucination moyenne après garde-fou : doit rester basse ; sinon le filet extractif a pris le relais.


In [3]:
from src.nlp.local_llm import gguf_path
p = gguf_path(paths.models)
print('GGUF local:', p.exists(), p)

GGUF local: True /Users/segalini-briant/Documents/GitHub/CISIA - Sujet Santé/models/gguf/qwen2.5-1.5b-instruct-q4_k_m.gguf
